# AI Research Trends Analysis

In this project, I am exploring recent AI research using data from the OpenAlex API.

My first API query returned some future-dated publications because I was sorting only by publication date. I therefore refined the query so that it only requests publications from 2020 up to today's date.

The main flow of the project is:

**OpenAlex API → raw JSON → selected fields → Python objects → DataFrame → validation → analysis → visualization**

### 1. Fetching data from the API

First, I fetch research publications from the OpenAlex API.

I use today's date as the end date so that the query does not intentionally request publications from the future. I also use `try/except` so that API or connection errors can be handled without stopping the whole program.

In [ ]:
import requests
from datetime import date

# Function to fetch AI research works from the OpenAlex API
def fetch_works(api_url):
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from API: {e}")
        return None

# Use today's date as the upper limit for publication dates
current_date = date.today().isoformat()

api_url = (
    "https://api.openalex.org/works"
        "?search=artificial%20intelligence"
        f"&filter=from_publication_date:2020-01-01,to_publication_date:{current_date}"
        "&per_page=100"
)
works_data = fetch_works(api_url)


### 2. Inspecting the API response

Before processing the data, I want to understand what the API actually returned.

I inspect the main data types, available keys, number of results, and a few fields from the first publication. This helps me understand the JSON structure before extracting the information I need.

In [ ]:
# Inspecting the structure of the API response

if works_data is not None:
    print(type(works_data))
    print(works_data.keys())
    print(type(works_data["results"]))
    print(len(works_data["results"]))
    print(works_data["results"][0]["title"])
    print(works_data["results"][0]["publication_year"])
    print(works_data["results"][0]["cited_by_count"])
    print(works_data["results"][0]["open_access"])

# The above inspection confirms that OpenAlex returns a dictionary. The actual research works are stored inside the `results` list.


<class 'dict'>
dict_keys(['meta', 'results', 'group_by'])
<class 'list'>
100
Artificial Intelligence in Education: A Review
2020
3963
{'is_oa': True, 'oa_status': 'gold', 'oa_url': 'https://doi.org/10.1109/access.2020.2988510', 'any_repository_has_fulltext': True}


### 3. Saving the raw API data

Before changing the data, I save the original API response as a JSON file.

This gives me a copy of the raw data exactly as it was received from OpenAlex, which is useful if I want to go back and inspect it later.

In [ ]:
import json

# saving the fetched raw data to a JSON file
def save_json(data, file_path):
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


if works_data is not None:
    save_json(works_data, "data/works_data.json")

### 4. Selecting the data I need

The OpenAlex response contains many nested fields, But I don't need all of them for this project.
For my analysis, I focus on four fields:

- title
- publication year
- citation count
- open-access status

Keeping only these fields makes the dataset easier to understand and analyze.

In [ ]:
# Function to extract the relevant information from the works data

def extract_relevant_fields(works_data):
    results = works_data["results"]

    extracted_data = []

    for work in results:
        extracted_data.append({
            "title": work["title"],
            "publication_year": work["publication_year"],
            "cited_by_count": work["cited_by_count"],
            "is_open_access": work["open_access"]["is_oa"]
            
        })
    return extracted_data

extracted_data = extract_relevant_fields(works_data)

print(len(extracted_data))
print(extracted_data[0])

100
{'title': 'Artificial Intelligence in Education: A Review', 'publication_year': 2020, 'cited_by_count': 3963, 'is_open_access': True}


### 5. Representing research works with OOP

After extracting the required fields, I also represent individual publications as Python objects.

The `ResearchWork` class contains information that is common to a research publication. The `OpenAlexWork` class inherits from it and adds the open-access status.

I still use Pandas later for analysing the full dataset, while the classes are used to model individual research works.

In [ ]:
# Parent class: Represent a general research publication.

class ResearchWork:
    def __init__(self, title, publication_year, cited_by_count):
        self.title = title
        self.publication_year = publication_year
        self.cited_by_count = cited_by_count

    def summary(self):
        return (
            f"{self.title} ({self.publication_year}) - "
            f"{self.cited_by_count} citations"
        )

#### Adding OpenAlex-specific information

`OpenAlexWork` inherits the common attributes and methods from `ResearchWork`.

I use `super()` to reuse the parent class constructor and then add the `is_open_access` attribute.

In [ ]:
# Child class: Represent a research work returned by OpenAlex
class OpenAlexWork(ResearchWork):
    def __init__(
        self,
        title,
        publication_year,
        cited_by_count,
        is_open_access
    ):
        super().__init__(
            title,
            publication_year,
            cited_by_count
        )

        self.is_open_access = is_open_access

#### Testing one research object

Before creating objects for the full dataset, I first test the classes using one publication.

This allows me to check that the attributes and inherited `summary()` method work as expected.

In [42]:
first_work = extracted_data[0]

paper = OpenAlexWork(
    first_work["title"],
    first_work["publication_year"],
    first_work["cited_by_count"],
    first_work["is_open_access"]
)

print(paper.title)
print(paper.publication_year)
print(paper.cited_by_count)
print(paper.is_open_access)
print(paper.summary())

Artificial Intelligence in Education: A Review
2020
3963
True
Artificial Intelligence in Education: A Review (2020) - 3963 citations


#### Creating objects for all publications

Now that the class works for one publication, I create an `OpenAlexWork` object for every item in the extracted dataset.

This gives me a list of research objects that represents the same publications in an object-oriented way.

In [43]:
research_objects = []

for work in extracted_data:
    research_objects.append(
        OpenAlexWork(
            work["title"],
            work["publication_year"],
            work["cited_by_count"],
            work["is_open_access"]
        )
    )

print(len(research_objects))
print(research_objects[0].summary())

100
Artificial Intelligence in Education: A Review (2020) - 3963 citations


### 6. Converting the data to a DataFrame

For analysing all publications together, I convert the extracted data into a Pandas DataFrame.

The objects are useful for representing individual research works, while the DataFrame makes operations such as counting, filtering and calculating averages much easier.

In [44]:
#converting to a dataframe

import pandas as pd

df = pd.DataFrame(extracted_data)

print(df.head())
df.info()

                                               title  publication_year  \
0     Artificial Intelligence in Education: A Review              2020   
1  The role of artificial intelligence in achievi...              2020   
2  Revolutionizing healthcare: the role of artifi...              2023   
3  BNAI, NO-TOKEN, and MIND-UNITY: Pillars of a S...              2022   
4  Artificial Intelligence and Management: The Au...              2021   

   cited_by_count  is_open_access  
0            3963            True  
1            3225            True  
2            3130            True  
3            4331            True  
4            2494            True  
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   title             100 non-null    str  
 1   publication_year  100 non-null    int64
 2   cited_by_count    100 non-null    int64
 3   is_open_access   

Validate the data

Before cleaning, inspect the dataset for:

- publication-year range
- missing values
- complete duplicate rows
- duplicate titles
- number of rows and columns

This helps decide what actually needs cleaning instead of applying cleaning steps blindly.

### 7. Validating the dataset

Before analysing the data, I check its quality.

I look at the dataset size, missing values, duplicate rows, duplicate titles, and the publication-year range. This helps me decide whether any cleaning is actually necessary instead of changing the data without a reason.

In [45]:
def validate_data(df):
    print("Data Validation")
    print("---------------")

    print(f"Shape: {df.shape}")

    print("\nMissing values per column:")
    print(df.isnull().sum())

    print(f"\nNumber of duplicated rows: {df.duplicated().sum()}")
    print(f"Number of duplicated titles: {df['title'].duplicated().sum()}")

    print(f"Minimum publication year: {df['publication_year'].min()}")
    print(f"Maximum publication year: {df['publication_year'].max()}")

validate_data(df)


Data Validation
---------------
Shape: (100, 4)

Missing values per column:
title               0
publication_year    0
cited_by_count      0
is_open_access      0
dtype: int64

Number of duplicated rows: 0
Number of duplicated titles: 0
Minimum publication year: 2020
Maximum publication year: 2024


#### Checking duplicate rows

The validation function tells me how many duplicates exist. I also display duplicate rows separately so that I could inspect them if any were found.

In [46]:
# Displaying all duplicate rows in the dataframe
duplicate_rows = df[df.duplicated(keep=False)]

display(duplicate_rows.head(10))

,title,publication_year,cited_by_count,is_open_access


### 8. Saving the prepared dataset

The validation did not find missing values or duplicate rows in the selected data, so no additional cleaning was needed.

I keep the original API response as JSON and save the selected, validated tabular data as CSV for the analysis that follows.

In [ ]:
# Function to save a DataFrame as CSV without the Pandas index

def save_csv(df, file_path):
    df.to_csv(
        file_path,
        index=False
    )

save_csv(df, "data/cleaned_works.csv")

### 9. Analysing the data

For the first analysis, I want to answer three simple questions:

1. How many publications are there in each year?
2. What is the average number of citations?
3. How many publications are open access compared with non-open access?

These results will also be used in the visualization section.

In [48]:
def get_publications_per_year(df):
    return (
        df["publication_year"]
        .value_counts()
        .sort_index()
    )


def get_average_citations(df):
    return df["cited_by_count"].mean()


def get_open_access_counts(df):
    return df["is_open_access"].value_counts()

#### Running the analysis

I now call the analysis functions and store their results in variables.

Keeping the results in variables means I can print them here and also reuse them later when I create charts.

In [49]:
publications_per_year = get_publications_per_year(df)

average_citations = get_average_citations(df)

open_access_counts = get_open_access_counts(df)

Finally, I print the calculated values so that I can review the results before moving on to visualization.

In [ ]:
print("Publications per year:")
print(publications_per_year)

print(f"\nAverage citations: {average_citations:.2f}")

print("\nOpen access counts:")
print(open_access_counts)

Publications per year:
publication_year
2020    38
2021    24
2022    16
2023    20
2024     2
Name: count, dtype: int64

Average citations: 1316.05

Open access counts:
is_open_access
True     87
False    13
Name: count, dtype: int64
